In [ ]:
!pip install python-igraph leidenalg

In [ ]:
!pip install plotly

#  Prerequisite: Data Generation

Before running the analysis of temporal data in this notebook, you must ensure that the network data has been generated.

**Please run the external script `youNIverse_graph.py` separately.**

You need to execute this script **10 times**, once for each year from **2010 to 2019**.

> **Note:** We have excluded this data generation step from this notebook because the algorithms are computationally expensive. Generating the full dataset takes a significant amount of time and is best done as a standalone process.

###  Batch Processing: Temporal Graph Analysis (2010-2019)

The following cell executes the core analysis pipeline for the entire decade. We iterate through each year and run the `results_temporal.py` script as a separate **subprocess**.
**Outputs Generated:**
For each year, this step calculates and saves:
* Community structures (Louvain/Leiden algorithms).
* Network metrics (Density, PageRank centrality).
* Serialized visualization data (`.pkl` files) required for the interactive dashboards below.

In [23]:
import subprocess
import sys
import os
import time

years_to_process = list(range(2010, 2020))

project_root = os.getcwd()
script_path = os.path.join(project_root, "src", "scripts", "results_temporal.py")

print(f"📂 Project Root: {project_root}")
print(f"📜 Script Path: {script_path}\n")

for year in years_to_process:
    print(f"⏳ Processing Analytics for Year {year}...")
    start_time = time.time()
    
    env = os.environ.copy()
    env["PYTHONPATH"] = f"{env.get('PYTHONPATH', '')}:{project_root}"
    

    try:
        result = subprocess.run(
            [sys.executable, "results_temporal.py", str(year)],
            cwd=os.path.join(project_root, "src", "scripts"),
            env=env,
            capture_output=True,
            text=True,
            check=True 
        )
        elapsed = time.time() - start_time
        print(f"✅ Year {year} finished in {elapsed:.2f} seconds.")


    except subprocess.CalledProcessError as e:
        print(f"❌ Error for Year {year}!")
        print(e.stderr) 
        break #

    print("-" * 50 + "\n")

📂 Project Root: /mloscratch/users/dafer/ada-2025-project-zlayjiya
📜 Script Path: /mloscratch/users/dafer/ada-2025-project-zlayjiya/src/scripts/results_temporal.py

⏳ Processing Analytics for Year 2010...
✅ Year 2010 finished in 7.08 seconds.
--------------------------------------------------

⏳ Processing Analytics for Year 2011...
✅ Year 2011 finished in 8.08 seconds.
--------------------------------------------------

⏳ Processing Analytics for Year 2012...
✅ Year 2012 finished in 8.99 seconds.
--------------------------------------------------

⏳ Processing Analytics for Year 2013...
✅ Year 2013 finished in 8.36 seconds.
--------------------------------------------------

⏳ Processing Analytics for Year 2014...
✅ Year 2014 finished in 9.34 seconds.
--------------------------------------------------

⏳ Processing Analytics for Year 2015...
✅ Year 2015 finished in 9.93 seconds.
--------------------------------------------------

⏳ Processing Analytics for Year 2016...
✅ Year 2016 fini

###  Network Fragmentation Analysis
The following script extracts the community count from our generated network snapshots to quantify the modularity over the decade.

**Key Observations:**
* **2010:** The network is highly centralized with only **16 major communities**.
* **Growth:** We observe a steady fragmentation, with a significant spike in 2016 corresponding to the massive influx of new users (the "Jio Effect").
* **2019:** The decade concludes with **99 distinct communities**, confirming the structural shift from a single "Town Square" to a fragmented "Archipelago."

In [22]:
import pickle
import os
import json
from IPython.display import HTML, display

# --- 1. CONFIGURATION ---
# Since the notebook is in the root folder, we access data directly
DATA_DIR = "data/results"
OUTPUT_DIR = "reports/figures"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "fragmentation.html")
YEARS = list(range(2010, 2020))

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. DATA EXTRACTION ---
print("--- Starting Fragmentation Analysis ---")

years_found = []
counts = []

for year in YEARS:
    # Path to the pickle file generated by your previous analysis
    pkl_path = os.path.join(DATA_DIR, str(year), f"viz_data_{year}.pkl")
    
    if not os.path.exists(pkl_path):
        print(f"⚠️  Year {year}: File not found at {pkl_path}. Skipping.")
        continue

    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)
            
            # Check if 'communities' key exists in the pickle data
            if 'communities' in data:
                count = len(data['communities'])
                years_found.append(year)
                counts.append(count)
                print(f"✅ Year {year}: {count} communities detected.")
            else:
                print(f"⚠️  Year {year}: 'communities' data missing.")

    except Exception as e:
        print(f"❌ Error processing year {year}: {e}")

# --- 3. CALCULATE GROWTH ---
# Calculate the percentage increase from start to finish
if len(counts) > 1:
    start_val = counts[0]
    end_val = counts[-1]
    if start_val > 0:
        growth_percent = int(((end_val - start_val) / start_val) * 100)
    else:
        growth_percent = 0
else:
    growth_percent = 0

# Prepare JSON data for injection into JavaScript
json_years = json.dumps(years_found)
json_counts = json.dumps(counts)

print(f"\n--- Global Growth: +{growth_percent}% ---")

# --- 4. HTML TEMPLATE GENERATION (DARK MODE) ---
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>YouTube Network Fragmentation | 2010-2019</title>
    <script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;700;900&display=swap" rel="stylesheet">
    <style>
        /* DARK MODE CSS */
        body {{ 
            font-family: 'Inter', sans-serif; 
            margin: 0; 
            background-color: transparent; 
            color: #e0e0e0;
        }}
        #chart-container {{ 
            width: 100%; 
            height: 600px; 
        }}
    </style>
</head>
<body>

    <div id="chart-container"></div>

<script>
    // --- DATA INJECTED FROM PYTHON ---
    const years = {json_years};
    const communitiesCount = {json_counts};
    const growthPercent = {growth_percent};

    // Configure the Scatter Trace
    const trace = {{
        x: years,
        y: communitiesCount,
        mode: 'lines+markers+text',
        name: 'Communities Count',
        text: communitiesCount,
        textposition: 'top center',
        textfont: {{ family: 'Inter', weight: 700, size: 13, color: '#e0e0e0' }},
        line: {{ color: '#e41a1c', width: 4, shape: 'spline' }}, // Red line
        marker: {{ 
            size: 10, 
            color: '#e41a1c', 
            line: {{ color: '#ffffff', width: 2 }} 
        }},
        fill: 'tozeroy', 
        fillcolor: 'rgba(228, 26, 28, 0.1)', // Subtle red fill
        type: 'scatter'
    }};

    // Historical Annotations logic
    const annotations = [
        // Main Growth Box (Top Left)
        {{
            xref: 'paper', yref: 'paper',
            x: 0.04, y: 0.98,
            text: `+${{growthPercent}}% Fragmentation`,
            showarrow: false,
            font: {{ family: 'Inter', size: 18, color: '#e41a1c', weight: 900 }},
            bgcolor: '#000000', // Black background for box
            bordercolor: '#e41a1c',
            borderwidth: 2,
            borderpad: 8
        }}
    ];

    // Helper to add timeline events safely
    const addEvent = (year, label, ax, ay) => {{
        const idx = years.indexOf(year);
        if (idx !== -1) {{
            annotations.push({{
                x: year, 
                y: communitiesCount[idx], 
                text: label, 
                showarrow: true, arrowhead: 2, ax: ax, ay: ay, 
                arrowcolor: '#e0e0e0',
                font: {{ size: 11, color: '#cccccc' }} 
            }});
        }}
    }};

    // Add specific historical events
    addEvent(2012, 'PewDiePie era starts', 0, 40);
    addEvent(2016, 'Massive Indian expansion (Jio)', 0, 40);
    addEvent(2019, 'Peak Network Modularity', -40, 0);

    const layout = {{
        title: {{
            text: 'YouTube Network Fragmentation (2010-2019)',
            font: {{ family: 'Inter', size: 24, weight: 900, color: '#ffffff' }},
            pad: {{ t: 20 }}
        }},
        font: {{
            family: 'Inter',
            color: '#e0e0e0'
        }},
        xaxis: {{
            title: 'Analysis Year',
            tickmode: 'linear',
            gridcolor: '#444444',
            zeroline: false,
            tickcolor: '#e0e0e0',
            font: {{ family: 'Inter', weight: 700 }}
        }},
        yaxis: {{
            title: 'Detected Communities (Louvain/Leiden)',
            gridcolor: '#444444',
            zeroline: false,
            tickcolor: '#e0e0e0',
            range: [0, Math.max(...communitiesCount) * 1.3]
        }},
        margin: {{ l: 80, r: 50, t: 100, b: 80 }},
        hovermode: 'x unified',
        annotations: annotations,
        paper_bgcolor: 'rgba(0,0,0,0)',
        plot_bgcolor: 'rgba(0,0,0,0)'
    }};

    // Render Plotly Chart
    Plotly.newPlot('chart-container', [trace], layout, {{responsive: true}});

</script>
</body>
</html>
"""

# --- 5. EXPORT AND DISPLAY ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"🎉 Dashboard saved to: {OUTPUT_FILE}")
print("👇 Displaying interactive chart below 👇")

# Display the HTML content directly in the notebook
display(HTML(html_content))

--- Starting Fragmentation Analysis ---
✅ Year 2010: 16 communities detected.
✅ Year 2011: 27 communities detected.
✅ Year 2012: 31 communities detected.
✅ Year 2013: 57 communities detected.
✅ Year 2014: 59 communities detected.
✅ Year 2015: 67 communities detected.
✅ Year 2016: 80 communities detected.
✅ Year 2017: 96 communities detected.
✅ Year 2018: 93 communities detected.
✅ Year 2019: 99 communities detected.

--- Global Growth: +518% ---
🎉 Dashboard saved to: reports/figures/fragmentation.html
👇 Displaying interactive chart below 👇


###  Network Density Analysis: The Connectivity Collapse
The following script calculates the **density** of the network for each year,defined as the ratio of actual edges to possible edges within the Largest Connected Component.

**Key Observations:**
* **2010 (Small World):** The decade begins with a density of **0.05**, meaning 5% of all possible connections between communities actually existed. This indicates a highly connected "global village."
* **The Collapse:** As the network scaled, density didn't just decrease; it plummeted.
* **2019 (Silos):** By the end of the decade, density flatlined at **0.002**. This represents a staggering **~95% drop** in relative connectivity, mathematically proving the emergence of filter bubbles where distinct groups no longer interact.

In [21]:
import pickle
import os
import json
import networkx as nx
from IPython.display import HTML, display

# --- 1. CONFIGURATION ---
DATA_DIR = "data/results"
OUTPUT_DIR = "reports/figures"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "density.html")
YEARS = list(range(2010, 2020))

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. DATA EXTRACTION ---
print("--- Starting Network Density Analysis ---")

years_found = []
densities = []

for year in YEARS:
    pkl_path = os.path.join(DATA_DIR, str(year), f"viz_data_{year}.pkl")
    
    if not os.path.exists(pkl_path):
        print(f"⚠️  Year {year}: File not found. Skipping.")
        continue

    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)
            
            # Ensure the Largest Connected Component (LCC) exists
            if 'LCC' in data:
                # Calculate Density: ratio of actual edges to possible edges
                d = nx.density(data['LCC'])
                
                years_found.append(year)
                densities.append(float(d))
                print(f"✅ Year {year}: Density {d:.6f}")
            else:
                print(f"⚠️  Year {year}: 'LCC' graph missing in pickle.")

    except Exception as e:
        print(f"❌ Error processing year {year}: {e}")

# --- 3. PREPARE DATA FOR VISUALIZATION ---
json_years = json.dumps(years_found)
json_densities = json.dumps(densities)

# --- 4. HTML TEMPLATE GENERATION ---
# Using the specific design provided
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>YouNiverse | Global Connectivity Decay</title>
    <script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;700;900&display=swap" rel="stylesheet">
    <style>
        /* DARK MODE CSS */
        body {{ 
            font-family: 'Inter', sans-serif; 
            margin: 0; 
            background-color: transparent; 
            color: #e0e0e0; 
            display: flex; 
            justify-content: center; 
            align-items: center; 
            height: 100vh; 
        }}
        
        #chart-container {{ 
            background: transparent; 
            border: none; 
            border-radius: none; 
            box-shadow: 0 15px 35px rgba(0,0,0,0.5); 
            width: 100%; 
            height: 600px; 
            padding: 20px; 
            box-sizing: border-box;
        }}
    </style>
</head>
<body>

    <div id="chart-container"></div>

<script>
    // --- DATA INJECTED FROM PYTHON ---
    const years = {json_years};
    const densities = {json_densities};

    // Trace Configuration
    const trace = {{
        x: years,
        y: densities,
        mode: 'lines+markers',
        name: 'Network Density',
        // Cyan/Blue color to pop against the dark background
        line: {{ color: '#00d2ff', width: 4, shape: 'spline' }}, 
        marker: {{ 
            size: 10, 
            color: '#00d2ff', 
            line: {{ color: '#ffffff', width: 2 }} 
        }},
        fill: 'tozeroy',
        fillcolor: 'rgba(0, 210, 255, 0.1)', // Subtle glowing effect
        type: 'scatter',
        hovertemplate: '<b>Year %{{x}}</b><br>Density: %{{y:.6f}}<extra></extra>'
    }};

    const layout = {{
        title: {{ 
            text: 'Decline of Global Connectivity (2010-2019)', 
            font: {{ family: 'Inter', size: 24, weight: 900, color: '#ffffff' }} 
        }},
        font: {{
            family: 'Inter',
            color: '#e0e0e0'
        }},
        xaxis: {{ 
            title: 'Year', 
            gridcolor: '#444444', 
            zeroline: false, 
            tickmode: 'linear',
            tickcolor: '#e0e0e0'
        }},
        yaxis: {{ 
            title: 'Structural Density', 
            gridcolor: '#444444', 
            zeroline: false, 
            tickformat: '.4f',
            tickcolor: '#e0e0e0'
        }},
        margin: {{ l: 80, r: 50, t: 100, b: 80 }},
        paper_bgcolor: 'rgba(0,0,0,0)',
        plot_bgcolor: 'rgba(0,0,0,0)',
        
        annotations: []
    }};

    Plotly.newPlot('chart-container', [trace], layout, {{responsive: true}});
</script>
</body>
</html>
"""

# --- 5. EXPORT AND DISPLAY ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"\n🎉 Dashboard saved to: {OUTPUT_FILE}")
print("👇 Displaying interactive chart below 👇")

display(HTML(html_content))

--- Starting Network Density Analysis ---
✅ Year 2010: Density 0.049970
✅ Year 2011: Density 0.025531
✅ Year 2012: Density 0.016567
✅ Year 2013: Density 0.010808
✅ Year 2014: Density 0.008324
✅ Year 2015: Density 0.006401
✅ Year 2016: Density 0.004465
✅ Year 2017: Density 0.003108
✅ Year 2018: Density 0.002410
✅ Year 2019: Density 0.002456

🎉 Dashboard saved to: reports/figures/density.html
👇 Displaying interactive chart below 👇


###  Dynamic PageRank Evolution: The Shift in Power
This visualization tracks the **PageRank Centrality** of major categories over the decade. PageRank measures the structural influence of a category—how often users flow into it from other parts of the network.

**Key Observations:**
* **2010 (The MTV Era):** Music (Red line) is the dominant gravitational center of YouTube.
* **2012 (The Flippening):** A historic crossover occurs where **Gaming** (Green line) overtakes Music. This marks the transition from a "video repository" to a community-driven creator platform.
* **Diversification:** While Music declines in structural importance (migrating to streaming services), categories like **Entertainment** and **People & Blogs** see steady growth, reflecting the rise of vloggers and native YouTube formats.

In [20]:
import os
from IPython.display import HTML, display

# --- 1. CONFIGURATION ---
OUTPUT_DIR = "reports/figures"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "pagerank.html")

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. HTML CONTENT DEFINITION ---
# We inject the provided HTML code directly
html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>YouNiverse | Dynamic PageRank Evolution</title>
    <script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;700;900&display=swap" rel="stylesheet">
    <style>
        /* DARK MODE CSS */
        body { 
            font-family: 'Inter', sans-serif; 
            margin: 0; 
            background-color: transparent; 
            color: #e0e0e0; 
            display: flex; 
            flex-direction: column; 
            align-items: center; 
            justify-content: center; 
            min-height: 100vh; 
            padding: 20px; 
            box-sizing: border-box;
        }
        
        #chart-container { 
            background: transparent;
            border: none; 
            border-radius: 24px; 
            box-shadow: none; 
            width: 95vw; 
            max-width: 1300px; 
            height: 85vh; 
            padding: 30px; 
            box-sizing: border-box; 
        }
    </style>
</head>
<body>

    <div id="chart-container"></div>

<script>
    // --- DATA INJECTED FROM PYTHON ---
    const years = [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019];
    const trends = {"Music": [0.31852134716913083, 0.30553552577074217, 0.20936813472797294, 0.16734175166263174, 0.1415728370399084, 0.1321792822639465, 0.11321981516497955, 0.11766109256574656, 0.12136776534613178, 0.0882868154017503], "Gaming": [0.09675335100473527, 0.17207552502767962, 0.2409544556481643, 0.31092040721448994, 0.31092941498915905, 0.30608518860492906, 0.30049858321197226, 0.2269554687678615, 0.21531248154223198, 0.259930612860708], "Entertainment": [0.17664782899072404, 0.1531111519972608, 0.17883274076692138, 0.18574299662231192, 0.20798918462025795, 0.19923244854396555, 0.2339984664970175, 0.2806352633618812, 0.27492531658704017, 0.2499529134888926], "People & Blogs": [0.05993605797475055, 0.0555918707936731, 0.06046270236432752, 0.0628057841538051, 0.07716704047624778, 0.09515786924750769, 0.08999702292165812, 0.10296060867856345, 0.09013475579036473, 0.0867038653527128], "News & Politics": [0.03015271949026299, 0.03073497347107335, 0.031740808580495654, 0.023205043244854593, 0.024548714455952435, 0.026704053161445092, 0.03311871499337016, 0.030714985677378043, 0.04621281031254372, 0.07752810214721247]};

    const colors = {
        "Music": "#e41a1c",
        "Gaming": "#4daf4a",
        "Entertainment": "#377eb8",
        "People & Blogs": "#984ea3",
        "News & Politics": "#ff7f00"
    };

    // Map categories to Plotly traces
    const traces = Object.keys(trends).map(category => {
        return {
            x: years,
            y: trends[category],
            name: category,
            mode: 'lines+markers',
            line: { shape: 'spline', width: 4, color: colors[category] || '#999' },
            marker: { size: 8 },
            hovertemplate: `<b>${category}</b><br>Centrality: %{y:.4f}<extra></extra>`
        };
    });

    const layout = {
        title: { 
            text: 'Evolution of Structural Influence (PageRank Sum)', 
            font: { family: 'Inter', size: 22, weight: 900, color: '#ffffff' } 
        },
        font: {
            family: 'Inter',
            color: '#e0e0e0'
        },
        xaxis: { 
            title: 'Year', 
            gridcolor: '#444444', 
            zeroline: false,
            tickmode: 'linear',
            tickcolor: '#e0e0e0'
        },
        yaxis: { 
            title: 'Total PageRank Centrality', 
            gridcolor: '#444444', 
            zeroline: false,
            tickcolor: '#e0e0e0'
        },
        legend: { 
            orientation: 'h', 
            y: -0.15, 
            x: 0.5, 
            xanchor: 'center',
            font: { color: '#e0e0e0' }
        },
        margin: { l: 80, r: 50, t: 80, b: 100 },
        hovermode: 'x unified',
        paper_bgcolor: 'rgba(0,0,0,0)', 
        plot_bgcolor: 'rgba(0,0,0,0)',  
        annotations: [] 
    };

    // Dynamic annotations
    if (years.includes(2012) && trends['Gaming']) {
        const idx = years.indexOf(2012);
        layout.annotations.push({
            x: 2012, y: trends['Gaming'][idx],
            text: 'Gaming Era', 
            showarrow: true, 
            arrowhead: 2, 
            ax: -40, ay: -40,
            arrowcolor: '#e0e0e0',
            font: { color: '#ffffff' }
        });
    }

    Plotly.newPlot('chart-container', traces, layout, {responsive: true});

</script>
</body>
</html>
"""

# --- 3. EXPORT AND DISPLAY ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"🎉 Dashboard saved to: {OUTPUT_FILE}")
print("👇 Displaying interactive chart below 👇")

# Display directly in the notebook
display(HTML(html_content))

🎉 Dashboard saved to: reports/figures/pagerank.html
👇 Displaying interactive chart below 👇


###  Top 5 Community Identities: The Rise of Parallel Worlds
This analysis tracks the **five largest communities** detected in the network for each year. For each community, we identify its "Identity" by looking at the dominant category and the channel with the highest PageRank.

**Key Observations:**
* **2010-2012:** The landscape is dominated by **Music** (Red bars). VEVO channels and major artists form the largest clusters.
* **The Gaming Takeover:** By 2013-2014, **Gaming** communities (Green/Blue bars) led by creators like PewDiePie explode in size, becoming the new heavyweights of the platform.
* **The Jio Effect (2016+):** A massive new entity appears—the **T-Series** community. Unlike Western gaming communities, this cluster grows in isolation, eventually becoming one of the largest forces on the platform by 2019, illustrating the "Parallel Worlds" phenomenon.

In [18]:
import pickle
import os
import json
import pandas as pd
from IPython.display import HTML, display

# --- 1. CONFIGURATION ---
DATA_DIR = "data/results"
OUTPUT_DIR = "reports/figures"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "top5_identities.html")
YEARS = list(range(2010, 2020))

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. DATA EXTRACTION ---
print("--- Starting Top 5 Communities Analysis ---")

# Structure to hold data for Rank 1 to Rank 5 across all years
final_data = {
    "years": YEARS,
    "ranks": {f"rank{i}": {"sizes": [], "labels": []} for i in range(1, 6)}
}

for year in YEARS:
    pkl_path = os.path.join(DATA_DIR, str(year), f"viz_data_{year}.pkl")
    
    # Handle missing years (fill with zeros)
    if not os.path.exists(pkl_path):
        print(f"⚠️  Year {year}: File not found. Filling with zeros.")
        for i in range(1, 6):
            final_data["ranks"][f"rank{i}"]["sizes"].append(0)
            final_data["ranks"][f"rank{i}"]["labels"].append("No Data")
        continue

    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)
            
            # Check if necessary keys exist
            if 'comm_summary' not in data or 'node_df' not in data:
                print(f"⚠️  Year {year}: Missing 'comm_summary' or 'node_df'.")
                continue

            comm_summary = data['comm_summary']
            node_df = data['node_df']

            # Get the top 5 largest communities by node count
            top5_comm = comm_summary.nlargest(5, 'n_nodes')

            # Loop through Rank 1 to 5
            for i in range(5):
                rank_key = f"rank{i+1}"
                
                # If a community exists for this rank
                if i < len(top5_comm):
                    row = top5_comm.iloc[i]
                    comm_id = row['community']
                    size = int(row['n_nodes'])
                    
                    # --- Determine Identity ---
                    # Filter nodes belonging to this community
                    members = node_df[node_df['community'] == comm_id]
                    
                    if not members.empty:
                        # 1. Dominant Category (Mode)
                        main_cat = members['category_cc'].mode()
                        main_cat = main_cat.iloc[0] if not main_cat.empty else "Unknown"
                        
                        # 2. Top Creator (Max PageRank)
                        if 'pagerank' in members.columns:
                            top_creator = members.nlargest(1, 'pagerank')['name_cc'].iloc[0]
                        else:
                            top_creator = "Unknown"
                    else:
                        main_cat = "Unknown"
                        top_creator = "Unknown"

                    # Create the label for the tooltip
                    label = f"{main_cat} ({top_creator})"
                    
                    final_data["ranks"][rank_key]["sizes"].append(size)
                    final_data["ranks"][rank_key]["labels"].append(label)
                
                # If fewer than 5 communities exist in that year
                else:
                    final_data["ranks"][rank_key]["sizes"].append(0)
                    final_data["ranks"][rank_key]["labels"].append("N/A")

            print(f"✅ Year {year}: Processed successfully.")

    except Exception as e:
        print(f"❌ Error processing year {year}: {e}")

# Prepare JSON data for injection
json_years = json.dumps(final_data["years"])
json_ranks = json.dumps(final_data["ranks"])

# --- 3. HTML TEMPLATE GENERATION ---
html_content = f"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Dynamic Top 5 Communities | YouNiverse</title>
    <script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;700;900&display=swap" rel="stylesheet">
    <style>
        /* DARK MODE CSS */
        body {{ 
            font-family: 'Inter', sans-serif; 
            margin: 0; 
            background-color: transparent; 
            color: #e0e0e0; 
            display: flex; 
            justify-content: center; 
            align-items: center; 
            height: 100vh; 
        }}
        
        #chart-container {{ 
            background: transparent; 
            border: none; 
            border-radius: 20px; 
            box-shadow: 0 10px 30px rgba(0,0,0,0.5); 
            width: 100%; 
            height: 600px; 
            padding: 20px; 
            box-sizing: border-box;
        }}
    </style>
</head>
<body>
    <div id="chart-container"></div>

<script>
    // --- DATA INJECTED FROM PYTHON ---
    const years = {json_years};
    const ranks = {json_ranks};

    // Colors for Rank 1 to Rank 5
    const colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3', '#ff7f00'];
    const rankLabels = ['#1 Largest', '#2 Largest', '#3 Largest', '#4 Largest', '#5 Largest'];

    // Construct Traces
    const traces = Object.keys(ranks).map((key, index) => {{
        return {{
            x: years,
            y: ranks[key].sizes,
            name: rankLabels[index],
            text: ranks[key].labels, // This contains "Category (Creator)"
            hovertemplate: '<b>Rank ' + (index+1) + '</b><br>%{{text}}<br>Size: %{{y}} channels<extra></extra>',
            type: 'bar',
            marker: {{ color: colors[index], opacity: 0.9 }}
        }};
    }});

    const layout = {{
        title: {{ 
            text: 'Evolution of Top 5 Community Identities', 
            font: {{ family: 'Inter', size: 22, weight: 900, color: '#ffffff' }} 
        }},
        font: {{
            family: 'Inter',
            color: '#e0e0e0'
        }},
        barmode: 'group', // Bars side by side
        xaxis: {{ 
            title: 'Year', 
            gridcolor: '#444444', 
            zeroline: false,
            tickcolor: '#e0e0e0'
        }},
        yaxis: {{ 
            title: 'Community Size (# of channels)', 
            gridcolor: '#444444', 
            zeroline: false,
            tickcolor: '#e0e0e0'
        }},
        legend: {{ 
            x: 0.01, 
            y: 0.99, 
            bgcolor: 'rgba(0, 0, 0, 0.5)', 
            font: {{ color: '#e0e0e0' }}
        }},
        margin: {{ l: 80, r: 50, t: 100, b: 80 }},
        hovermode: 'x unified',
        paper_bgcolor: 'rgba(0,0,0,0)', 
        plot_bgcolor: 'rgba(0,0,0,0)'
    }};

    Plotly.newPlot('chart-container', traces, layout, {{responsive: true}});
</script>
</body>
</html>
"""

# --- 4. EXPORT AND DISPLAY ---
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"\n🎉 Dashboard saved to: {OUTPUT_FILE}")
print("👇 Displaying interactive chart below 👇")

display(HTML(html_content))

--- Starting Top 5 Communities Analysis ---
✅ Year 2010: Processed successfully.
✅ Year 2011: Processed successfully.
✅ Year 2012: Processed successfully.
✅ Year 2013: Processed successfully.
✅ Year 2014: Processed successfully.
✅ Year 2015: Processed successfully.
✅ Year 2016: Processed successfully.
✅ Year 2017: Processed successfully.
✅ Year 2018: Processed successfully.
✅ Year 2019: Processed successfully.

🎉 Dashboard saved to: reports/figures/top5_identities.html
👇 Displaying interactive chart below 👇


###  The Galaxy Explorer: A Macro-View of the Universe
This final visualization aggregates the raw graph data into a simplified "Galaxy Map."
* **Nodes:** Each bubble represents a detected community (a "Galaxy"). Its size is proportional to the number of channels inside.
* **Edges:** The lines represent the flow of users between these galaxies.
* **Goal:** This interactive tool allows us to observe the structural drift from a single cohesive mass (2010) to a dispersed archipelago of distinct subcultures (2019).

In [19]:
import pickle
import json
import os
import networkx as nx
from collections import defaultdict
from IPython.display import IFrame, display

# --- 1. CONFIGURATION ---
# Path to your data folder (using data/results from root)
DATA_DIR = "data/results" 
OUTPUT_DIR = "reports/figures"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "galaxy_explorer.html")
YEARS = list(range(2010, 2020))

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 2. DATA PROCESSING FUNCTION ---
def process_year(year):
    """
    Reads the PKL file for a specific year and aggregates channels into galaxies.
    Returns a dictionary with 'nodes' (communities) and 'links' (inter-community edges).
    """
    pkl_path = os.path.join(DATA_DIR, str(year), f"viz_data_{year}.pkl")
    
    if not os.path.exists(pkl_path):
        print(f"⚠️  Year {year}: File not found. Skipping.")
        return None

    try:
        with open(pkl_path, "rb") as f:
            data = pickle.load(f)

        LCC = data['LCC']
        communities = data['communities']
        
        # Load node metadata if available
        if 'node_df' in data:
            node_df = data['node_df'].set_index("channel_id").to_dict(orient="index")
        else:
            node_df = {}

        # --- A. PREPARE NODES (GALAXIES) ---
        nodes_json = []
        node2comm = {} 
        comm_sizes = {} 
        
        # Calculate node strength (weighted degree) for ranking
        strength = {
            u: sum(d.get("weight", d.get("weight_raw", 1)) for _, d in LCC[u].items()) 
            for u in LCC.nodes()
        }

        # Iterate through communities
        for cid, nodes_c in enumerate(communities):
            # Filter out tiny micro-communities to keep the viz clean
            if len(nodes_c) < 5: continue
            
            for node in nodes_c:
                node2comm[node] = cid
            
            # Identify the top 5 channels in this community to name it
            sorted_nodes = sorted(nodes_c, key=lambda u: strength.get(u, 0), reverse=True)[:5]
            
            top_channels = []
            for n in sorted_nodes:
                info = node_df.get(n, {})
                # Use name_cc (Title Case) or channel ID as fallback
                name = info.get('name_cc') or str(n)
                top_channels.append(name)
            
            comm_id = f"comm_{cid}"
            comm_sizes[comm_id] = len(nodes_c)
            
            nodes_json.append({
                "id": comm_id,
                # Name the galaxy after its largest channel
                "name": top_channels[0] if top_channels else f"Galaxy {cid}",
                "size": len(nodes_c),
                "top_channels": top_channels
            })

        # --- B. PREPARE EDGES (GRAVITY) ---
        comm_edges = defaultdict(int)
        
        # Aggregate raw edges into community-to-community edges
        for u, v, d in LCC.edges(data=True):
            cu = node2comm.get(u)
            cv = node2comm.get(v)
            
            # Only count edges between DIFFERENT communities
            if cu is not None and cv is not None and cu != cv:
                edge_key = tuple(sorted([cu, cv]))
                weight = d.get("weight", d.get("weight_raw", 1))
                comm_edges[edge_key] += weight

        links_json = []
        for (c1, c2), w in comm_edges.items():
            id1 = f"comm_{c1}"
            id2 = f"comm_{c2}"
            
            # Calculate density (normalized weight)
            if id1 in comm_sizes and id2 in comm_sizes:
                size_product = comm_sizes[id1] * comm_sizes[id2]
                # Scale factor for visualization
                density = (w / size_product) * 1000 if size_product > 0 else 0
                
                links_json.append({
                    "source": id1,
                    "target": id2,
                    "weight": float(density),
                    "raw_weight": int(w)
                })

        print(f"✅ Year {year}: {len(nodes_json)} galaxies, {len(links_json)} connections.")
        return {"nodes": nodes_json, "links": links_json}

    except Exception as e:
        print(f"❌ Error processing {year}: {e}")
        return None

# --- 3. EXECUTE PROCESSING ---
print("--- Starting Galaxy Extraction ---")
all_data = {}

for year in YEARS:
    result = process_year(year)
    if result:
        all_data[year] = result

json_payload = json.dumps(all_data)

print("--- Generating HTML Explorer ---")

# --- 4. HTML TEMPLATE (D3.js) ---
html_template = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>YouNiverse | Galaxy Explorer</title>
    <script src="https://d3js.org/d3.v7.min.js"></script>
    <link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600;800&display=swap" rel="stylesheet">
    <style>
        :root { 
            --primary: #00d2ff; 
            --bg: #000000; 
            --text: #e0e0e0; 
            --panel: rgba(20, 20, 20, 0.95); 
            --border: #333;
        }
        
        body { 
            font-family: 'Inter', sans-serif; 
            background-color: var(--bg); 
            color: var(--text); 
            overflow: hidden; 
            margin: 0; 
        }
        
        #viz-layout { display: flex; width: 100vw; height: 100vh; position: relative; }

        #controls { position: absolute; top: 20px; left: 20px; z-index: 1000; display: flex; flex-direction: column; gap: 10px; }
        
        .btn { 
            padding: 12px 20px; 
            background: #1a1a1a; 
            color: white;
            border: 1px solid var(--border); 
            border-radius: 30px; 
            cursor: pointer; 
            font-weight: 700; 
            font-family: 'Inter'; 
            transition: 0.3s; 
            box-shadow: 0 4px 6px rgba(0,0,0,0.5); 
        }
        .btn:hover { background: #333; border-color: var(--primary); }

        #info-panel {
            position: absolute; top: 20px; right: 20px; width: 340px; 
            background: var(--panel);
            border: 1px solid var(--border); 
            border-radius: 12px; padding: 24px;
            box-shadow: 0 10px 30px rgba(0,0,0,0.5); 
            z-index: 100; display: none;
            backdrop-filter: blur(10px);
        }
        #info-panel.active { display: block; }
        #info-panel h2 { margin-top: 0; color: #fff; }

        #timeline-container {
            position: absolute; bottom: 30px; left: 50%; transform: translateX(-50%);
            width: 450px; background: #1a1a1a; padding: 20px 30px; border-radius: 60px;
            border: 1px solid var(--border);
            box-shadow: 0 5px 20px rgba(0,0,0,0.5); display: flex; align-items: center; gap: 20px; z-index: 1000;
        }
        #year-slider { flex-grow: 1; accent-color: var(--primary); cursor: pointer; }
        #year-display { font-weight: 900; font-size: 22px; color: var(--primary); min-width: 60px; }

        /* --- UPDATED EDGE STYLES --- */
        .link { 
            stroke: #555; 
            stroke-opacity: 0.2; 
            transition: stroke 0.3s, stroke-opacity 0.3s, filter 0.3s; 
            pointer-events: none; 
        }

        /* --- HIGHLIGHT STYLES --- */
        .edge-highlight {
            stroke: var(--primary) !important;
            stroke-opacity: 0.8 !important;
            filter: drop-shadow(0 0 4px var(--primary));
            z-index: 500;
        }

        .node circle { stroke: #fff; stroke-width: 1.5px; cursor: pointer; transition: all 0.2s; }
        .node text { 
            font-size: 11px; font-weight: 700; fill: #bbb; 
            pointer-events: none; text-shadow: 0 2px 4px #000; 
        }
        
        .faded { opacity: 0.05 !important; }
        
        .highlight { stroke: var(--primary) !important; stroke-width: 4px !important; filter: drop-shadow(0 0 8px var(--primary)); }
        
        #top-channels-list { list-style: none; padding: 0; }
        #top-channels-list li { 
            background: #222; margin-bottom: 8px; padding: 12px; 
            border-radius: 8px; font-size: 13px; font-weight: 600; color: #eee; 
            border: 1px solid #333;
        }
    </style>
</head>
<body>

<div id="viz-layout">
    <div id="controls">
        <button id="recenter-btn" class="btn">Recenter View</button>
    </div>
    
    <div id="timeline-container">
        <span id="year-display">2010</span>
        <input type="range" id="year-slider" min="2010" max="2019" step="1" value="2010">
    </div>

    <div id="info-panel">
        <h2 id="comm-name">Galaxy Name</h2>
        <p id="comm-size" style="color: #888;">0 Channels</p>
        <hr style="border-color: #333;">
        <p style="color: var(--primary); text-transform: uppercase; font-size: 12px; font-weight: 800; letter-spacing: 1px;">Top 5 Leaders</p>
        <ul id="top-channels-list"></ul>
    </div>
</div>

<script>
    // --- INJECTED DATA PLACEHOLDER ---
    const dataset = __JSON_DATA_PLACEHOLDER__;
    
    const width = window.innerWidth, height = window.innerHeight;
    
    const svg = d3.select("#viz-layout").append("svg")
        .attr("width", width)
        .attr("height", height)
        .style("background-color", "var(--bg)");
        
    const g = svg.append("g");

    const zoom = d3.zoom()
        .scaleExtent([0.1, 8])
        .on("zoom", (e) => g.attr("transform", e.transform));
    svg.call(zoom);

    let currentYear = 2010;
    let simulation;
    
    const colorScale = d3.scaleOrdinal(d3.schemeTableau10);

    function updateGraph(year) {
        currentYear = year;
        
        g.selectAll("*").remove();
        if (simulation) simulation.stop();
        d3.select("#info-panel").classed("active", false);

        const data = dataset[year];

        if (!data) {
            g.append("text")
                .attr("x", width/2).attr("y", height/2)
                .attr("fill", "white").attr("text-anchor", "middle")
                .style("font-size", "24px")
                .text("No data available for " + year);
            return;
        }

        const nodes = data.nodes.map(d => ({...d}));
        const links = data.links.map(d => ({...d}));

        const sizeScale = d3.scaleSqrt()
            .domain(d3.extent(nodes, d => d.size))
            .range([12, 70]);
        
        const weightExtent = d3.extent(links, d => d.weight);
        const distanceScale = d3.scaleLinear()
            .domain(weightExtent)
            .range([400, 100]); 

        simulation = d3.forceSimulation(nodes)
            .force("link", d3.forceLink(links).id(d => d.id).distance(d => distanceScale(d.weight)))
            .force("charge", d3.forceManyBody().strength(d => -1000 - (d.size * 5)))
            .force("center", d3.forceCenter(width / 2, height / 2))
            .force("collide", d3.forceCollide().radius(d => sizeScale(d.size) + 15).iterations(2));

        // --- DRAW EDGES ---
        const link = g.append("g")
            .selectAll("line")
            .data(links)
            .join("line")
            .attr("class", "link")
            .attr("stroke-width", d => Math.min(Math.log10(d.raw_weight + 1) * 3, 8));

        // --- DRAW NODES ---
        const node = g.append("g")
            .selectAll("g")
            .data(nodes)
            .join("g")
            .attr("class", "node")
            .call(d3.drag()
                .on("start", dragstarted)
                .on("drag", dragged)
                .on("end", dragended));

        node.append("circle")
            .attr("r", d => sizeScale(d.size))
            .attr("fill", (d, i) => colorScale(i % 10))
            .attr("fill-opacity", 0.9);

        node.append("text")
            .attr("text-anchor", "middle")
            .attr("dy", d => sizeScale(d.size) + 20)
            .text(d => d.name);

        // --- INTERACTION ---
        node.on("click", (event, d) => {
            event.stopPropagation();
            
            d3.select("#info-panel").classed("active", true);
            d3.select("#comm-name").text(d.name + " Cluster");
            d3.select("#comm-size").text(`${d.size.toLocaleString()} Channels`);
            
            const list = d3.select("#top-channels-list").html("");
            d.top_channels.forEach(channelName => {
                list.append("li").text(channelName);
            });

            // Visual Logic: Focus on selected node
            node.classed("faded", n => n.id !== d.id);
            link.classed("faded", l => l.source.id !== d.id && l.target.id !== d.id);
            link.classed("edge-highlight", l => l.source.id === d.id || l.target.id === d.id);
            
            d3.selectAll("circle").classed("highlight", false);
            d3.select(event.currentTarget).select("circle").classed("highlight", true);
        });

        simulation.on("tick", () => {
            link
                .attr("x1", d => d.source.x)
                .attr("y1", d => d.source.y)
                .attr("x2", d => d.target.x)
                .attr("y2", d => d.target.y);

            node.attr("transform", d => `translate(${d.x},${d.y})`);
        });
    }

    document.getElementById("year-slider").oninput = function() {
        document.getElementById("year-display").innerText = this.value;
        updateGraph(this.value);
    };

    document.getElementById("recenter-btn").onclick = () => {
        svg.transition().duration(750).call(zoom.transform, d3.zoomIdentity);
    };

    // Reset when clicking background
    svg.on("click", () => {
        d3.select("#info-panel").classed("active", false);
        d3.selectAll(".faded").classed("faded", false);
        d3.selectAll(".highlight").classed("highlight", false);
        d3.selectAll(".edge-highlight").classed("edge-highlight", false);
    });

    function dragstarted(event, d) { if (!event.active) simulation.alphaTarget(0.3).restart(); d.fx = d.x; d.fy = d.y; }
    function dragged(event, d) { d.fx = event.x; d.fy = event.y; }
    function dragended(event, d) { if (!event.active) simulation.alphaTarget(0); d.fx = null; d.fy = null; }

    updateGraph(2010);
</script>
</body>
</html>
"""

# --- 5. INJECTION & WRITE FILE ---
final_html = html_template.replace("__JSON_DATA_PLACEHOLDER__", json_payload)

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(final_html)

print(f"\n🎉 Success! Galaxy Explorer generated at: {OUTPUT_FILE}")
print("👇 Displaying interactive chart below (IFrame) 👇")

# Display using IFrame for better D3 isolation
display(IFrame(src=OUTPUT_FILE, width="100%", height="800px"))

--- Starting Galaxy Extraction ---
✅ Year 2010: 12 galaxies, 56 connections.
✅ Year 2011: 19 galaxies, 105 connections.
✅ Year 2012: 22 galaxies, 145 connections.
✅ Year 2013: 32 galaxies, 210 connections.
✅ Year 2014: 36 galaxies, 260 connections.
✅ Year 2015: 44 galaxies, 300 connections.
✅ Year 2016: 52 galaxies, 297 connections.
✅ Year 2017: 64 galaxies, 362 connections.
✅ Year 2018: 60 galaxies, 316 connections.
✅ Year 2019: 66 galaxies, 289 connections.
--- Generating HTML Explorer ---

🎉 Success! Galaxy Explorer generated at: reports/figures/galaxy_explorer.html
👇 Displaying interactive chart below (IFrame) 👇
